# How many replicates does a check need?

Reads the ten replicates of one configuration written by

```bash
python experiments/exploration-replicate-variability/run.py
```

`replication-reporting`, ten examples from ten distinct documents, ten
identical runs. The note is
[`thinking/experiments/exploration-replicate-variability.md`](../../thinking/experiments/exploration-replicate-variability.md)
— read it first: it records that this probe aims deliberately at the
noisiest check, so the number here is closer to an upper bound than a
typical value.

> **This notebook has been rewritten to work per property.**
>
> The earlier version reduced each replicate to a single
> instance-weighted mean **across** leaf properties. That weighting was a
> workaround for `mean_score` returning `0.0` when a property had nothing
> applicable — a defect since fixed at the source, where the mean is now
> `None` and carries its denominator.
>
> The workaround was tolerable for sizing a replicate count, which is all
> this probe was for. It is not carried into benchmarking: mixing
> `panel_label` with `explanation` produces a number that belongs to
> neither, and it turns out to hide the main finding here — the noise is
> concentrated in one property, which a pooled figure cannot show.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from soda_mmqc.core.run_layout import iter_leaves
from soda_mmqc.core.scoring import ANALYSIS_FILENAME, score_check
from soda_mmqc.reporting import load_run_root, replicate_spread, scores_frame

RUNS = Path("../../experiments/runs/exploration-replicate-variability").resolve()
CHECKLIST, CHECK = "fig-checklist-exp01", "replication-reporting"
FULL_BENCHMARK = 38   # examples exp-01 will run; this probe uses ten

assert RUNS.is_dir(), f"no runs at {RUNS}; run the probe first"
sorted(f"{arm}/rep-{rep:02d}" for arm, rep, _ in iter_leaves(RUNS))

## Score every leaf, then read one tidy frame

`score_check` takes one `rep-NN/`. The scored `analysis.json` is derived
data and deliberately not committed, so it is produced here — cheaply,
locally, and skipped for any leaf already scored.

In [ ]:
for arm, replicate, leaf in iter_leaves(RUNS):
    if not (leaf / ANALYSIS_FILENAME).is_file():
        score_check(CHECKLIST, CHECK, leaf, save=True)

runs = load_run_root(RUNS, checklist=CHECKLIST, check=CHECK)
scores = scores_frame(runs)

print(f"{len(scores)} rows = "
      f"{scores['replicate'].nunique()} replicates x "
      f"{scores['example'].nunique()} examples x "
      f"{scores['property'].nunique()} properties")
scores.head()

## Where the replicate noise actually is

One row per property: the mean over replicates, and the SD over
replicates with examples collapsed within each replicate first. This is
resampling noise — "how much would this move if we ran it again" — not
example-to-example variation.

`n_scored_total` is the denominator. Properties differ in how often they
are applicable at all, and a mean over 500 instances is not comparable to
a mean over 800.

In [ ]:
spread = replicate_spread(scores).sort_values("sd", ascending=False)
spread[["property", "mean", "sd", "n_replicates", "n_scored_total"]]

This is the finding the pooled version could not show: the noise is not
spread evenly across the check. Read the top row against the bottom
before treating "the check" as having one variance.

## The decomposition, per property

The quantity exp-01 needs is the spread of a **property mean over 38
examples**. This probe has ten, so the observed spread must not be read as
exp-01's — a mean over ten examples is noisier by about √(38/10) ≈ 2.

Instead estimate the *per-example* between-replicate variance, which does
not depend on how many examples were run, and derive the check-level
spread from it:

$$\mathrm{SD}(\bar{s}_N) = \sqrt{\bar{v}/N}, \qquad
\bar{v} = \text{mean per-example variance}$$

In [ ]:
def decompose(per_property: pd.DataFrame) -> pd.Series:
    """Per-example variance across replicates, averaged over examples."""
    grid = per_property.pivot_table(
        index="example", columns="replicate", values="mean_score"
    )
    per_example_var = grid.astype(float).var(axis=1, ddof=1)
    return pd.Series(
        {
            "n_examples": int(grid.shape[0]),
            "v_bar": float(per_example_var.mean()),
            "per_example_sd": float(np.sqrt(per_example_var.mean())),
            # SD of the ten observed replicate means, for the check below
            "observed_sd_at_10": float(
                grid.astype(float).mean(axis=0).std(ddof=1)
            ),
        }
    )


decomposition = (
    scores.groupby("property")[scores.columns.tolist()]
    .apply(decompose, include_groups=False)
    .sort_values("per_example_sd", ascending=False)
)
decomposition

### Does the extrapolation hold?

It assumes examples vary independently. Test it per property: the derived
spread at N=10 should match the spread actually observed across the ten
replicate means. Where they disagree materially, examples are correlated
and the step to 38 is not valid for that property — say so in the note
rather than using the number.

In [ ]:
check = decomposition.copy()
check["derived_sd_at_10"] = np.sqrt(check["v_bar"] / check["n_examples"])
check["ratio"] = check["observed_sd_at_10"] / check["derived_sd_at_10"]
check["independence_ok"] = check["ratio"].between(0.7, 1.4)
check[["observed_sd_at_10", "derived_sd_at_10", "ratio", "independence_ok"]]

## What a replicate count buys, per property

`SE(n) = SD(mean over 38) / √n`. Compare against the difference exp-01 is
looking for: where detailed-minus-minimal is smaller than `SE(3)` for a
property, three replicates will not resolve *that property*, whatever they
do for the others.

In [ ]:
sd_38 = np.sqrt(decomposition["v_bar"] / FULL_BENCHMARK)

table = pd.DataFrame(
    {n: sd_38 / np.sqrt(n) for n in (1, 3, 5, 7, 10)},
)
table.columns = [f"SE at n={n}" for n in table.columns]
table.insert(0, "SD over 38 examples", sd_38)
table.sort_values("SD over 38 examples", ascending=False).round(4)

The paired difference exp-01 actually measures has a lower SE than a
single arm's, because pairing by example cancels whatever noise is common
to both arms. These columns are therefore an upper bound on what it
faces; only exp-01 itself can say by how much.

## Non-response

A replicate that returned nothing scores as a fully missing row set rather
than being excluded. Even one changes how the averages above should be
read.

In [ ]:
from soda_mmqc.reporting import non_response_counts

nr = non_response_counts(runs)
empty = nr[nr["empty"]]
print(f"{len(empty)} empty answer(s) in {len(nr)} example-runs")
empty[["replicate", "example", "correct_row", "missing_row"]]

## Cost

In [ ]:
import json as _json

usage = [
    {"usd": u["total_cost_usd"], "turns": u.get("num_turns"),
     "seconds": (u.get("duration_ms") or 0) / 1000}
    for audit in sorted(RUNS.rglob("tool_audit.json"))
    for u in [(_json.loads(audit.read_text()).get("usage") or {})]
    if u.get("total_cost_usd") is not None
]
spend = pd.DataFrame(usage)
if len(spend):
    print(f"{len(spend)} sessions, ${spend['usd'].sum():.2f} total, "
          f"${spend['usd'].mean():.4f} mean")
    spend.describe().round(4)
else:
    print("no usage recorded")